# **＜重要＞本章で使用するファイル**

- real_estate.csv

これのファイルを左サイドバーのファイルからアップロードしてお使いください。（p.20参照）

Google ドライブにファイルを置き、マウントして使うこともできます（p.21参照）。その場合は次の点に注意してください。

- ファイルのパスは、各自のフォルダに合わせて書き換えてください
- ドライブ上のファイルは各自で管理してください
- ファイルを書き込むコード（to_csv() など）を実行すると、パスを誤ったときに既存のファイルを上書きするおそれがあります

# Menu.53 機械学習モデルが苦手なデータを理解しよう

データの全体像を把握する

In [ ]:
import pandas as pd

df = pd.read_csv("real_estate.csv")
print(df.head())
print(df.info())

基本統計量を確認する

In [ ]:
print(df.describe())

# Menu.54 正規化で数値を0〜1の範囲に揃えよう

データを正規化する

In [ ]:
import numpy as np
from sklearn.preprocessing import MinMaxScaler

# シンプルなデータで試す
area_data = np.array([[50], [70], [100]])
print("元データ")
print(area_data)

# スケーラーを作成
scaler = MinMaxScaler()

# fit(): データから最小値･最大値を学習
scaler.fit(area_data)

# transform(): 学習した情報を使って変換
area_normalized = scaler.transform(area_data)
print("正規化後")
print(area_normalized)

学習と正規化を一度に行う

In [ ]:
# fit_transform()で一度に実行
scaler = MinMaxScaler()
area_normalized = scaler.fit_transform(area_data)

スケーラーを再利用する

In [ ]:
new_data = np.array([[45], [65], [105]])

# transform()で変換(fit済みのスケーラーを再利用)
new_data_normalized = scaler.transform(new_data)
print("正規化後")
print(new_data_normalized)

データを正規化する

In [ ]:
import pandas as pd
from sklearn.preprocessing import MinMaxScaler
from sklearn.model_selection import train_test_split

df = pd.read_csv("real_estate.csv")

# 説明変数と目的変数の準備
X = df[["area", "age", "distance_station"]]
y = df["price"]

# データ分割
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

# MinMaxScalerの作成と正規化
scaler = MinMaxScaler()
X_train_normalized = scaler.fit_transform(X_train)  # 訓練データで基準を作成
X_test_normalized = scaler.transform(X_test)  # 同じ基準でテストデータを変換

# DataFrameに変換
X_train_normalized_df = pd.DataFrame(
    X_train_normalized,
    columns=["area", "age", "distance_station"]
)

print("正規化前")
print(X_train.describe())
print("正規化後")
print(X_train_normalized_df.describe())

# Menu.55 標準化で平均0・標準偏差1に揃えよう

データを標準化する

In [ ]:
from sklearn.preprocessing import StandardScaler

# StandardScalerの作成
scaler = StandardScaler()

# 訓練データで標準化(fit_transform)
X_train_scaled = scaler.fit_transform(X_train)

# テストデータで標準化(transform)
X_test_scaled = scaler.transform(X_test)

# DataFrameに変換(列名を維持)
X_train_scaled_df = pd.DataFrame(
    X_train_scaled,
    columns=["area", "age", "distance_station"]
)

print("標準化後")
print(X_train_scaled_df.describe())

# Menu.56 エンコーディングでカテゴリ変数を数値化しよう

ダミー変数化する

In [ ]:
import pandas as pd
df = pd.read_csv("real_estate.csv")

# カテゴリ変数のダミー変数化
df_encoded = pd.get_dummies(
    df,
    columns=["region", "direction"],
    drop_first=True # 多重共線性を避けるため、最初の1つを省略
)

print(df_encoded.head(3))

# Menu.57 線形回帰分析を理解しよう

重回帰モデルを構築する

In [ ]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LinearRegression

# データ読み込み
df = pd.read_csv("real_estate.csv")

# ダミー変数化
df_encoded = pd.get_dummies(
    df,
    columns=["region", "direction"],
    drop_first=True
)

# 説明変数と目的変数の準備
X = df_encoded.drop(columns=["id", "price"])
y = df_encoded["price"]

# データ分割
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

# 標準化
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# モデルの構築と学習
model = LinearRegression()
model.fit(X_train_scaled, y_train)

モデルのパラメータを確認する

In [ ]:
# 学習したパラメータの確認
print("切片", model.intercept_)

# 係数を見やすく整理
for i in range(len(X.columns)):
  print(X.columns[i], model.coef_[i])

MAEを評価する

In [ ]:
from sklearn.metrics import mean_absolute_error

# 単回帰モデル(面積のみ)用に標準化
scaler_simple = StandardScaler()
X_train_area = scaler_simple.fit_transform(X_train[["area"]])
X_test_area = scaler_simple.transform(X_test[["area"]])

model_simple = LinearRegression()
model_simple.fit(X_train_area, y_train)
y_test_pred_simple = model_simple.predict(X_test_area)
mae_simple = mean_absolute_error(y_test, y_test_pred_simple)

# 重回帰モデル(複数の変数)
y_test_pred = model.predict(X_test_scaled)
mae_multiple = mean_absolute_error(y_test, y_test_pred)

print("単回帰(面積のみ)のMAE", mae_simple)
print("重回帰(複数変数)のMAE", mae_multiple)

# Menu.58 RMSEとR²スコアでモデルを評価しよう

RMSEでモデルを評価する

In [ ]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LinearRegression
from sklearn.metrics import root_mean_squared_error

# データ読み込み
df = pd.read_csv("real_estate.csv")

# ダミー変数化
df_encoded = pd.get_dummies(df, columns=["region", "direction"],drop_first=True)

# 説明変数と目的変数
X = df_encoded.drop(columns=["id", "price"])
y = df_encoded["price"]

# データ分割
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2,random_state=42)

# 標準化
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# モデルの学習と予測
model = LinearRegression()
model.fit(X_train_scaled, y_train)
y_test_pred = model.predict(X_test_scaled)

# RMSEの計算
rmse_test = root_mean_squared_error(y_test, y_test_pred)

print("RMSE", rmse_test)

R²スコアを計算する

In [ ]:
from sklearn.metrics import r2_score
# R2スコアの計算
r2_test = r2_score(y_test, y_test_pred)
print("R2スコア", r2_test)

# Menu.60 正則化で過学習を防ごう

Lasso回帰モデルを構築する

In [ ]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import Lasso
from sklearn.metrics import mean_absolute_error

# データ読み込み
df = pd.read_csv("real_estate.csv")

# ダミー変数化
df_encoded = pd.get_dummies(df, columns=["region", "direction"],drop_first=True)

# 説明変数と目的変数
X = df_encoded.drop(columns=["id", "price"])
y = df_encoded["price"]

# データ分割
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2,random_state=42)

# 標準化
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# Lassoモデルの構築と学習
lasso = Lasso(alpha=100)
lasso.fit(X_train_scaled, y_train)

# 予測と評価
y_test_pred = lasso.predict(X_test_scaled)
mae_test = mean_absolute_error(y_test, y_test_pred)
print("MAE", mae_test)

# 係数を確認
print("切片", lasso.intercept_)
for i in range(len(X.columns)):
  print(X.columns[i], lasso.coef_[i])

Ridge回帰モデルを構築する

In [ ]:
from sklearn.linear_model import Ridge

# Ridgeモデルの構築と学習
ridge = Ridge(alpha=100)
ridge.fit(X_train_scaled, y_train)

# 予測と評価
y_test_pred = ridge.predict(X_test_scaled)
mae_test = mean_absolute_error(y_test, y_test_pred)
print("MAE", mae_test)

# 係数を確認
print("切片", ridge.intercept_)
for i in range(len(X.columns)):
  print(X.columns[i], ridge.coef_[i])

# Menu.61 決定木で複雑なルールを学ぼう

決定木回帰モデルを構築する

In [ ]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeRegressor
from sklearn.metrics import mean_absolute_error

# データ読み込み
df = pd.read_csv("real_estate.csv")
# ダミー変数化

df_encoded = pd.get_dummies(df, columns=["region", "direction"],drop_first=True)

# 説明変数と目的変数
X = df_encoded.drop(columns=["id", "price"])
y = df_encoded["price"]

# データ分割
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2,random_state=42)

# DecisionTreeRegressorモデルの構築
dt = DecisionTreeRegressor(
    max_depth=5, # 木の深さの上限
    min_samples_split=50, # 分岐に必要な最小サンプル数
    min_samples_leaf=20, # 葉ノードの最小サンプル数
    random_state=42
)

# 学習(決定木はスケーリング不要)
dt.fit(X_train, y_train)

# 予測
y_test_pred_dt = dt.predict(X_test)

# 評価
mae_test_dt = mean_absolute_error(y_test, y_test_pred_dt)

print("MAE", mae_test_dt)

決定木を可視化する

In [ ]:
from sklearn.tree import plot_tree
import matplotlib.pyplot as plt

plt.figure(figsize=(30, 10))
plot_tree(dt,
          feature_names=X.columns,
          filled=True,
          rounded=True,
          fontsize=10)

plt.tight_layout()
plt.show()

# Menu.62 ランダムフォレストで高精度な予測を実現しよう

ランダムフォレストモデルを構築する

In [ ]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error

# データ読み込み
df = pd.read_csv("real_estate.csv")

# ダミー変数化
df_encoded = pd.get_dummies(df, columns=["region", "direction"],drop_first=True)

# 説明変数と目的変数
X = df_encoded.drop(columns=["id", "price"])
y = df_encoded["price"]

# データ分割
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2,random_state=42)

# RandomForestRegressorモデルの構築
rf = RandomForestRegressor(
    n_estimators=100, # 決定木の本数(100本の木で｢森｣を作る)
    max_depth=10, # 各木の深さの上限
    min_samples_split=20, # 分岐に必要な最小サンプル数
    min_samples_leaf=10, # 葉ノードの最小サンプル数
    random_state=42,
    n_jobs=-1 # 全CPUコアを使用
)

# 学習
rf.fit(X_train, y_train)

# 予測と評価
y_test_pred_rf = rf.predict(X_test)
mae_test_rf = mean_absolute_error(y_test, y_test_pred_rf)

print("MAE", mae_test_rf)

特徴量の重要度を確認する

In [ ]:
for i in range(len(X.columns)):
  print(X.columns[i], rf.feature_importances_[i])

# Menu.63 交差検証でモデルの真の実力を測ろう

交差検証を実装する

In [ ]:
import pandas as pd
from sklearn.model_selection import cross_val_score
from sklearn.ensemble import RandomForestRegressor

# データ読み込み
df = pd.read_csv("real_estate.csv")

# ダミー変数化
df_encoded = pd.get_dummies(df, columns=["region", "direction"],drop_first=True)

# 説明変数と目的変数
X = df_encoded.drop(columns=["id", "price"])
y = df_encoded["price"]

# ランダムフォレストモデルの構築
rf = RandomForestRegressor(
    n_estimators=100,
    max_depth=10,
    random_state=42,
    n_jobs=-1
)

# 5-fold交差検証を実行
# scoring="neg_mean_absolute_error"は負の値を返す(scikit-learnの仕様)
cv_scores = cross_val_score(
    rf, # モデル
    X, # 説明変数(全データ)
    y, # 目的変数(全データ)
    cv=5, # 5分割
    scoring="neg_mean_absolute_error" # 評価指標
)

# 負のMAEを正のMAEに変換
cv_scores = -cv_scores

print("平均MAE", cv_scores.mean())
print("標準偏差", cv_scores.std())

4つのモデルを交差検証する

In [ ]:
from sklearn.linear_model import LinearRegression, Ridge
from sklearn.tree import DecisionTreeRegressor

# 複数のモデルを準備
models = {
    "線形回帰": LinearRegression(),
    "Ridge回帰": Ridge(alpha=1.0),
    "決定木": DecisionTreeRegressor(max_depth=10, random_state=42),
    "ランダムフォレスト": RandomForestRegressor(n_estimators=100,max_depth=10, random_state=42)
}

# 各モデルで交差検証を実行
for name, model in models.items():
  cv_scores = cross_val_score(
      model, X, y,
      cv=5,
      scoring="neg_mean_absolute_error",
      n_jobs=-1
  )
  cv_scores = -cv_scores

  print(name, cv_scores.mean())